# 🎯 Query Enhancement: 
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

### Types of Query Enhancements:
- [1. Query Expansion Technique](#1-query-expansion-technique)
- [2. Query Decomposition](#2-query-decomposition)
- [3. HyDE](#3-hyde)

### 1. Query Expansion Technique:
In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be. That's where Query Expansion becomes very handy.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [3]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [4]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [5]:
for i, chunk in enumerate(chunks[:2]):
    print(f"\n📄Chunk {i+1}:")
    print(f"    Source: {chunk.metadata['source']}")
    print(f"    Length: {len(chunk.page_content)} characters")
    print(f"    Content: {chunk.page_content[:100]}...")  # Print first 100 characters
    print("-" * 40)


📄Chunk 1:
    Source: langchain_crewai_dataset.txt
    Length: 299 characters
    Content: LangChain is an open-source framework designed for developing applications powered by large language...
----------------------------------------

📄Chunk 2:
    Source: langchain_crewai_dataset.txt
    Length: 172 characters
    Content: and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LL...
----------------------------------------


In [6]:
### step 2: Vector Store
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [7]:
### step 3: MMR Retriever
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5})

In [8]:
## step 4 : LLM and Prompt
llm=init_chat_model("openai:o4-mini")

In [9]:
# Query expansion prompt
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")
query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()

In [10]:
# test the query expansion chain
query_expansion_chain.invoke({"query":"Langchain memory"})

'“LangChain memory” OR “Lang Chain memory” OR “LangChain conversational memory” OR “LangChain state management” OR “LangChain memory module” OR “LangChain memory buffer” OR “LangChain memory interface” OR “LangChain memory backend” OR “LangChain memory store” OR “LangChain memory persistence” OR “LangChain short-term vs long-term memory” OR “LangChain session memory” OR “LangChain chat history” OR “LangChain context window” OR “LangChain stateful agents” OR “memory management in LangChain” OR “LangChain RAG memory” OR “LangChain retrieval-augmented generation memory” OR “vector store memory in LangChain” OR “LangChain embeddings cache” OR “LangChain Redis memory store” OR “LangChain FAISS/Chroma/Pinecone memory” OR “LangChain PostgreSQL memory plugin” OR “LangChain DynamoDB memory” OR “LangChain memory best practices” OR “LangChain memory tutorial” OR “LangChain memory code examples” OR “LangChain memory Python” OR “LangChain memory Node.js” OR “chatbot memory architecture” OR “AI agen

In [11]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = answer_prompt | llm

In [12]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"], # pyright: ignore[reportIndexIssue]
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]})) # pyright: ignore[reportIndexIssue]
    })
    | document_chain
)

In [13]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:  
“What types of memory architectures, modules, and storage backends does LangChain support for managing conversational or application state? Include both short-term/context-window memory (e.g. BufferMemory, ConversationBufferMemory, ConversationBufferWindowMemory), summary or condensation memory (e.g. ConversationSummaryMemory), and long-term or persistent memory implementations (e.g. SQLMemory, RedisMemory, DynamoDBMemory, vector-store-based Memory using FAISS, Pinecone, Chroma, Weaviate, Milvus, Weaviate, etc.). Also cover related technical terms and use cases such as session state, chat history persistence, embeddings-based retrieval memory, retrieval-augmented generation (RAG), context windowing, memory chaining, external vector stores, and semantic context storage.”
✅ Answer:
 content='LangChain currently provides two primary memory modules:  \n- ConversationBufferMemory – which retains the full dialogue history, and  \n- ConversationSummaryMemory – which keeps a 

In [14]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:  
(CrewAI OR “Crew AI” OR “CrewAI agents” OR “Crew AI agents” OR “crew management AI” OR “digital crew coordination” OR “autonomous crew scheduling system”)  
AND (agent OR agents OR bot OR bots OR “software agent” OR “autonomous agent”)  
AND (scheduling OR rostering OR “personnel assignment” OR “resource allocation” OR “crew coordination” OR “shift planning” OR “roster optimization”)  
AND (AI OR “artificial intelligence” OR “machine learning” OR “reinforcement learning” OR “multi-agent system” OR “constraint programming” OR “heuristic optimization”)  
AND (aviation OR airline OR maritime OR shipping OR space OR “offshore operations” OR “field service” OR healthcare OR “emergency response”)  

Synonyms & related terms included:  
• Crew management agents, AI-driven crew bots, autonomous scheduling assistants  
• Personnel assignment agents, workforce optimization AI, digital roster planners  
• Multi-agent systems, RL-based scheduling, constraint-based rostering
✅ Ans

### 2. Query Decomposition:
🧠 Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

In [16]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [17]:
### step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [18]:
### step 2: Vector Store and retriever
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
### step 3: LLM and Prompt
llm=init_chat_model(model="groq:llama-3.1-8b-instant")

In [23]:
# Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [24]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})

In [30]:
print(decomposition_question)

Here are some sub-questions to decompose the complex question:

1. **How does LangChain implement memory in its models?**
2. **What are the different types of agents supported by LangChain, and how do they utilize memory?**
3. **How does CrewAI handle memory management in its models?** 
4. **What are the key differences in agent capabilities and memory integration between LangChain and CrewAI?** 


These sub-questions break down the comparison into specific aspects of memory and agent functionality in each framework, allowing for more targeted document retrieval. 



In [25]:
# QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [26]:
# Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):

    # Step 1: Decompose the query into sub-questions
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]

    # Step 2: Retrieve documents and get answers for each sub-question
    results = []
    for sub_q in sub_questions:
        docs = retriever.invoke(sub_q)
        result = qa_chain.invoke({"input": sub_q, "context": docs})
        results.append(f"Q: {sub_q}\nA: {result}")

    # Step 3: Combine answers into a final response
    final_answer = "\n\n".join(results)
    return final_answer


In [27]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: To better understand and retrieve relevant documents, I can decompose the complex question into the following smaller sub-questions:
A: It seems you're discussing the capabilities of LangChain, a tool for building AI applications.

To better understand and retrieve relevant documents, you can decompose the complex question into the following smaller sub-questions, enabling semantic search within large document corpora using LangChain's integration with vector databases like FAISS, Chroma, Pinecone, and Weaviate.

Q: **What is LangChain and what is its primary use case?**
A: Based on the provided context, LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). 

Its primary use case is to simplify the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration.

Q: This sub-question will help me understand the context of La

### 3. HyDE
🧠 What is HyDe?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user’s query directly, you first generate a hypothetical answer (document) to the query using an LLM — and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short.
2. Language mismatch between query and documents. 
3. You want to retrieve based on answer content, not question words


In [28]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma, FAISS
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

In [30]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [31]:
# 1. Load and chunk your dataset
chunk_size = 300
chunk_overlap = 100

# loading data
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()

c:\Users\mrinm\Downloads\MyTraining\GenerativeAI\RAG\RAG_Bootcamp_Udemy\.venv\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\mrinm\Downloads\MyTraining\GenerativeAI\RAG\RAG_Bootcamp_Udemy\.venv\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [32]:
# text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
chunks = text_splitter.split_documents(documents=documents)

In [33]:
for i, chunk in enumerate(chunks[:2]):
    print(f"\n📄Chunk {i+1}:")
    print(f"    Source: {chunk.metadata['source']}")
    print(f"    Length: {len(chunk.page_content)} characters")
    print(f"    Content: {chunk.page_content[:100]}...")  # Print first 100 characters
    print("-" * 40)


📄Chunk 1:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs
    Length: 291 characters
    Content: Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, co-inventor, an...
----------------------------------------

📄Chunk 2:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs
    Length: 298 characters
    Content: Apple Inc. (as Apple Computer Company) with Steve Wozniak and Ronald Wayne in 1976. After the compan...
----------------------------------------


In [34]:
# 2. Build vector store
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma.from_documents(documents = chunks,embedding=embedding_model,persist_directory = "output/steve_jobs_for_hyde.db")

##create the retriever
base_retriever=db.as_retriever(search_kwargs = {"k":5})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
# 3. Set up the LLM you’ll use to generate hypothetical answers
llm = init_chat_model("groq:llama-3.3-70b-versatile")

In [36]:
## Generating a prompt gor generating HyDE
def get_hyde_doc(query) -> str:
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template = template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query = query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hypo_doc = response.content
    return str(hypo_doc)

In [37]:
query = 'When was Steve Jobs fired from Apple?'
print(get_hyde_doc(query=query))

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
The infamous departure of Steve Jobs from Apple in 1985 is a pivotal moment in the history of the tech industry. To set the context, it's essential to delve into the events leading up to his ousting and the circumstances surrounding his eventual return to the company he co-founded.

In the early 1980s, Apple was experiencing a period of rapid growth and innovation, driven in large part by the success of the Apple II computer. However, the company's leadership was also becoming increasingly fragmented, with Jobs and then-CEO John Sculley holding differing visions for the future of Apple.

Jobs, who had co-founded Apple with Steve Wozniak in 1976, had become a dominant force within the company, driving the development of the Macintosh computer. However, his merc

In [38]:
matched_docs = base_retriever.invoke(get_hyde_doc(query))
for i, doc in enumerate(matched_docs):
	print(f"\n📄Matched Document {i+1}:")
	print(f"    Source: {doc.metadata.get('source', doc.metadata.get('title', 'N/A'))}")
	print(f"    Length: {len(doc.page_content)} characters")
	print(f"    Content: {doc.page_content[:200]}...")  # Print first 200 characters
	print("-" * 40)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]

📄Matched Document 1:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs
    Length: 297 characters
    Content: In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That same year, Jobs took some Apple employees with him to found NeXT, a computer plat...
----------------------------------------

📄Matched Document 2:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs
    Length: 297 characters
    Content: In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That same year, Jobs took some Apple employees with him to found NeXT, a computer plat...
----------------------------------------

📄Matched Document 3:
    Source: https://en.wikip

#### 3.1. Langchain-HypotheticalDocumentEmbedder

In [39]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [40]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [41]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [42]:
# Step 1: Load and split documents
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [51]:
# Step 2: Set up LLM and embeddings
base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = init_chat_model("groq:llama-3.3-70b-versatile")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


According to the official documentation and LangChain source code (mapping in PROMPT_MAP), the default options are:

- web_search
- sci_fact
- arguana
- trec_covid
- fiqa
- dbpedia_entity
- trec_news
- mr_tydi

In [52]:
# Step 3: HyDE Embedder using prompt_key='web_search'
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    prompt_key="web_search"
)

In [53]:
# Step 4: Store documents in Chroma with HyDE embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,
    persist_directory="output/langchain_hyde"
)

In [54]:
# Step 5: RAG answer generation prompt
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [55]:
# Step 6: Final RAG Pipeline
def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query, k=4)
    print(matched_docs)
    response = rag_chain.invoke({
        "input": query,
        "context": matched_docs
    })
    return response

In [56]:
# Step 7: Run example query
query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("✅ Final Answer:\n", answer)

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modular and composable. Components like retrievers, memories, agents, and chains can be easily combined and reused. This makes it ideal for building scalable, maintainable LLM applications. (v1)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modular and composable. Components like retrievers, memories, agents, and chains can be easily combined and reused. This makes it ideal for building scalable, maintainable LLM applications. (v1)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modular and composable. Components like retrievers, memories, agents, and chains can be easily combined and reused. This makes it ideal for building scalable, maintainable LLM applications. (v1)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modul

#### 3.2. Custom Prompt for Langchain HyDE

In [57]:
from langchain_core.prompts import PromptTemplate
custom = PromptTemplate.from_template(
    "Generate a concise hypothetical answer for this topic: {query}"
)

# Step 3: HyDE Embedder using prompt_key='web_search'
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=custom
)